# EDA 01 — Primeira passagem pelo dataset SRAG

## Objetivo

Realizar a primeira inspeção reproduzível do Parquet oficial do projeto, carregando somente as colunas necessárias.

Esta etapa verifica:

- acesso ao Parquet distribuído pela GitHub Release;
- versão e localização do arquivo;
- schema e colunas disponíveis;
- compatibilidade da lista de colunas da EDA;
- dimensões e consumo de memória;
- tipos de dados;
- período temporal;
- valores ausentes;
- cardinalidade;
- categorias observadas;
- possíveis duplicidades;
- possíveis variáveis posteriores à admissão hospitalar.

## Limites desta etapa

Este notebook não:

- remove registros;
- imputa valores;
- converte ausência em “não”;
- define a lista final de features;
- cria o alvo definitivo;
- realiza split de treino, validação e holdout;
- treina modelos;
- altera configurações ou gates.

As conclusões desta EDA são descritivas e não causais.


In [26]:
from pathlib import Path
import platform
import sys

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

from IPython.display import display


def encontrar_raiz_projeto() -> Path:
    """
    Localiza a raiz do projeto procurando pelas pastas src e configs.
    """
    pasta_atual = Path.cwd().resolve()

    for pasta in [pasta_atual, *pasta_atual.parents]:
        possui_src = (pasta / "src").is_dir()
        possui_configs = (pasta / "configs").is_dir()

        if possui_src and possui_configs:
            return pasta

    raise FileNotFoundError(
        "Não foi possível localizar a raiz do projeto."
    )


ROOT_DIR = encontrar_raiz_projeto()

if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))


from src.utils import config as C
from src.utils import env
from src.utils import storage


env.carregar_dotenv()

print(f"Raiz do projeto: {ROOT_DIR}")
print(f"Python: {sys.version.split()[0]}")
print(f"Sistema: {platform.system()} {platform.release()}")
print(f"pandas: {pd.__version__}")

Raiz do projeto: C:\Users\olivem85\OneDrive - Pfizer\Documents\GitHub\Projeto_Integrador
Python: 3.14.7
Sistema: Windows 11
pandas: 3.0.6


In [27]:
print(C.resumo())

print(f"Release configurada: {C.DATA_VERSION}")
print(f"Nome do arquivo: {C.PARQUET_NAME}")
print(f"Diretório de tabelas: {C.TABLES_DIR}")
print(f"Diretório de figuras: {C.FIGURES_DIR}")

ROOT_DIR          = C:\Users\olivem85\OneDrive - Pfizer\Documents\GitHub\Projeto_Integrador
DATA_DIR          = C:\Users\olivem85\OneDrive - Pfizer\Documents\GitHub\Projeto_Integrador\data
RAW_DIR           = C:\Users\olivem85\OneDrive - Pfizer\Documents\GitHub\Projeto_Integrador\data\raw
PROCESSED_DIR     = C:\Users\olivem85\OneDrive - Pfizer\Documents\GitHub\Projeto_Integrador\data\processed
ANALYTICAL_DIR    = C:\Users\olivem85\OneDrive - Pfizer\Documents\GitHub\Projeto_Integrador\data\analytical
TREINO_DIR        = C:\Users\olivem85\OneDrive - Pfizer\Documents\GitHub\Projeto_Integrador\data\treino
OUTPUTS_DIR       = C:\Users\olivem85\OneDrive - Pfizer\Documents\GitHub\Projeto_Integrador\outputs
RUNS_DIR          = C:\Users\olivem85\OneDrive - Pfizer\Documents\GitHub\Projeto_Integrador\outputs\runs
LOGS_DIR          = C:\Users\olivem85\OneDrive - Pfizer\Documents\GitHub\Projeto_Integrador\logs
CONFIGS_DIR       = C:\Users\olivem85\OneDrive - Pfizer\Documents\GitHub\Projeto_Integrad

In [28]:
COLUNAS_ESTRUTURA = [
    "nu_notific",
    "dt_notific",
    "dt_sin_pri",
    "sem_pri",
    "hospital",
    "dt_interna",
]

COLUNAS_CANDIDATAS_PRINCIPAIS = [
    "cs_sexo",
    "nu_idade_n",
    "tp_idade",
    "cs_gestant",
    "cs_raca",
    "cs_etinia",
    "cs_escol_n",
    "cs_zona",
    "ave_suino",
    "out_anim",
    "febre",
    "tosse",
    "garganta",
    "dispneia",
    "desc_resp",
    "saturacao",
    "diarreia",
    "vomito",
    "dor_abd",
    "fadiga",
    "perd_olft",
    "perd_pala",
    "puerpera",
    "cardiopati",
    "hematologi",
    "sind_down",
    "hepatica",
    "asma",
    "diabetes",
    "neurologic",
    "pneumopati",
    "imunodepre",
    "renal",
    "obesidade",
    "tabag",
]

COLUNAS_COMPLEMENTARES_TEXTO = [
    "outro_sin",
    "outro_des",
    "obes_imc",
    "out_morbi",
    "morb_desc",
    "out_antiv",
    "out_trat",
]

COLUNAS_GEOGRAFIA = [
    "sg_uf_not",
    "co_regiona",
    "co_mun_not",
    "co_pais",
    "sg_uf",
    "co_rg_resi",
    "co_mun_res",
    "sg_uf_inte",
    "co_rg_inte",
    "co_mu_inte",
]

COLUNAS_CONDICIONAIS_VACINACAO = [
    "vacina",
    "dt_ut_dose",
    "mae_vac",
    "dt_vac_mae",
    "m_amamenta",
    "dt_doseuni",
    "dt_1_dose",
    "dt_2_dose",
    "vacina_cov",
    "dose_1_cov",
    "dose_2_cov",
    "dose_ref",
    "dose_2ref",
    "dose_adic",
    "dos_re_bi",
]

COLUNAS_CONTEXTO_SOCIAL_CONDICIONAL = [
    "pac_cocbo",
    "pac_dscbo",
    "pov_ct",
    "tp_pov_ct",
    "estrang",
]

COLUNAS_TRATAMENTO_AUDITORIA = [
    "antiviral",
    "tp_antivir",
    "dt_antivir",
    "trat_cov",
    "tipo_trat",
    "dt_trt_cov",
]

COLUNAS_DESFECHO_AUDITORIA = [
    "evolucao",
    "dt_evoluca",
    "dt_encerra",
    "dt_digita",
]

COLUNAS_EDA = list(
    dict.fromkeys(
        COLUNAS_ESTRUTURA
        + COLUNAS_CANDIDATAS_PRINCIPAIS
        + COLUNAS_COMPLEMENTARES_TEXTO
        + COLUNAS_GEOGRAFIA
        + COLUNAS_CONDICIONAIS_VACINACAO
        + COLUNAS_CONTEXTO_SOCIAL_CONDICIONAL
        + COLUNAS_TRATAMENTO_AUDITORIA
        + COLUNAS_DESFECHO_AUDITORIA
    )
)

print(f"Total de colunas da EDA: {len(COLUNAS_EDA)}")
print(f"Colunas de estrutura: {len(COLUNAS_ESTRUTURA)}")
print(f"Colunas candidatas: {len(COLUNAS_CANDIDATAS_PRINCIPAIS)}")
print(f"Colunas complementares de texto: {len(COLUNAS_COMPLEMENTARES_TEXTO)}")
print(f"Colunas geográficas: {len(COLUNAS_GEOGRAFIA)}")
print(f"Colunas Condicionais de Vacinação: {len(COLUNAS_CONDICIONAIS_VACINACAO)}")
print(f"Colunas Contexto Social: {len(COLUNAS_CONTEXTO_SOCIAL_CONDICIONAL)}")
print(f"Colunas de auditoria: {len(COLUNAS_TRATAMENTO_AUDITORIA)}")
print(f"Colunas de Desfecho: {len(COLUNAS_DESFECHO_AUDITORIA)}")
print(f"Total sem duplicidades: {len(COLUNAS_EDA)}")

Total de colunas da EDA: 88
Colunas de estrutura: 6
Colunas candidatas: 35
Colunas complementares de texto: 7
Colunas geográficas: 10
Colunas Condicionais de Vacinação: 15
Colunas Contexto Social: 5
Colunas de auditoria: 6
Colunas de Desfecho: 4
Total sem duplicidades: 88


In [29]:
caminho_parquet = storage.get_parquet(force=False)

print(f"Arquivo: {caminho_parquet}")
print(f"Existe: {caminho_parquet.exists()}")
print(
    "Tamanho do arquivo: "
    f"{caminho_parquet.stat().st_size / 1024**2:,.2f} MB"
)

Arquivo: C:\Users\olivem85\OneDrive - Pfizer\Documents\GitHub\Projeto_Integrador\data\processed\dados-v2\srag.parquet
Existe: True
Tamanho do arquivo: 289.51 MB


In [30]:
arquivo_parquet = pq.ParquetFile(caminho_parquet)

schema = arquivo_parquet.schema_arrow
colunas_disponiveis = schema.names

metadados_parquet = {
    "arquivo": caminho_parquet.name,
    "versao_dados": C.DATA_VERSION,
    "numero_linhas": arquivo_parquet.metadata.num_rows,
    "numero_colunas": arquivo_parquet.metadata.num_columns,
    "numero_row_groups": arquivo_parquet.metadata.num_row_groups,
}

display(
    pd.DataFrame(
        [metadados_parquet]
    )
)

,arquivo,versao_dados,numero_linhas,numero_colunas,numero_row_groups
0,srag.parquet,dados-v2,4666910,198,28


In [31]:
schema_df = pd.DataFrame(
    {
        "coluna": schema.names,
        "tipo_arrow": [
            str(campo.type)
            for campo in schema
        ],
    }
)

display(schema_df.head(20))
print(f"Total de campos no schema: {len(schema_df)}")

,coluna,tipo_arrow
0,id_registro,int64
1,ano_arquivo,int16
2,data_snapshot,date32[day]
3,arquivo_origem,string
4,nu_notific,string
5,dt_notific,string
6,sem_not,string
7,dt_sin_pri,string
8,sem_pri,string
9,sg_uf_not,string


Total de campos no schema: 198


In [32]:
conjunto_colunas_disponiveis = set(colunas_disponiveis)

status_colunas = pd.DataFrame(
    {
        "coluna": COLUNAS_EDA,
        "grupo": (
            ["estrutura"] * len(COLUNAS_ESTRUTURA)
            + ["candidatas_principais"] * len(COLUNAS_CANDIDATAS_PRINCIPAIS)
            + ["complementares_texto"] * len(COLUNAS_COMPLEMENTARES_TEXTO)
            + ["geografia"] * len(COLUNAS_GEOGRAFIA)
            + ["condicionais_vacinacao"] * len(COLUNAS_CONDICIONAIS_VACINACAO)
            + ["contexto_social_condicional"] * len(COLUNAS_CONTEXTO_SOCIAL_CONDICIONAL)
            + ["tratamento_auditoria"] * len(COLUNAS_TRATAMENTO_AUDITORIA)
            + ["desfecho_auditoria"] * len(COLUNAS_DESFECHO_AUDITORIA)
        ),
    }
)

status_colunas["existe_no_parquet"] = (
    status_colunas["coluna"]
    .isin(conjunto_colunas_disponiveis)
)

display(status_colunas)

colunas_ausentes = status_colunas.loc[
    ~status_colunas["existe_no_parquet"],
    "coluna",
].tolist()

if colunas_ausentes:
    print("Colunas ausentes:")
    for coluna in colunas_ausentes:
        print(f"- {coluna}")
else:
    print("Todas as colunas solicitadas existem no Parquet.")

,coluna,grupo,existe_no_parquet
0,nu_notific,estrutura,True
1,dt_notific,estrutura,True
2,dt_sin_pri,estrutura,True
3,sem_pri,estrutura,True
4,hospital,estrutura,True
...,...,...,...
83,dt_trt_cov,tratamento_auditoria,True
84,evolucao,desfecho_auditoria,True
85,dt_evoluca,desfecho_auditoria,True
86,dt_encerra,desfecho_auditoria,True


Todas as colunas solicitadas existem no Parquet.


In [33]:
if colunas_ausentes:
    raise ValueError(
        "A leitura foi interrompida porque existem "
        "colunas ausentes no Parquet: "
        f"{colunas_ausentes}"
    )

print("Validação concluída. A leitura pode continuar.")

Validação concluída. A leitura pode continuar.


In [34]:
df = storage.read_srag(
    columns=COLUNAS_EDA,
)

print(f"Linhas carregadas: {df.shape[0]:,}")
print(f"Colunas carregadas: {df.shape[1]:,}")

display(df.head())

Linhas carregadas: 4,666,910
Colunas carregadas: 88


,nu_notific,dt_notific,dt_sin_pri,sem_pri,hospital,dt_interna,cs_sexo,nu_idade_n,tp_idade,cs_gestant,...,antiviral,tp_antivir,dt_antivir,trat_cov,tipo_trat,dt_trt_cov,evolucao,dt_evoluca,dt_encerra,dt_digita
0,31781201987679,2026-06-11,2026-06-02,23,1,2026-06-11,F,1,2,6,...,2,NaN,NaN,2,NaN,NaN,1,2026-06-19,2026-06-19,2026-06-11
1,31775652598724,2026-03-23,2026-03-22,13,NaN,NaN,F,75,3,5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-04-08
2,31781205405890,2026-03-29,2026-03-23,13,2,NaN,F,4,3,6,...,2,NaN,NaN,2,NaN,NaN,1,2026-03-31,2026-06-11,2026-06-11
3,31782407823707,2026-06-17,2026-06-15,25,1,2026-06-15,F,37,3,5,...,2,NaN,NaN,2,NaN,NaN,NaN,NaN,NaN,2026-06-25
4,31773493207022,2026-03-14,2026-03-09,11,1,2026-03-10,F,4,3,6,...,NaN,NaN,NaN,NaN,NaN,NaN,1,NaN,NaN,2026-03-14


In [35]:
if df.empty:
    raise ValueError(
        "O DataFrame foi carregado sem registros."
    )

if list(df.columns) != COLUNAS_EDA:
    print(
        "Atenção: a ordem das colunas carregadas "
        "difere da lista solicitada."
    )

print(f"Dimensão: {df.shape}")
print(f"Índice único: {df.index.is_unique}")
print(f"Quantidade de linhas: {len(df):,}")
print(f"Quantidade de colunas: {df.shape[1]:,}")

Dimensão: (4666910, 88)
Índice único: True
Quantidade de linhas: 4,666,910
Quantidade de colunas: 88


In [36]:
tipos_memoria = pd.DataFrame(
    {
        "coluna": df.columns,
        "tipo_pandas": [
            str(df[coluna].dtype)
            for coluna in df.columns
        ],
        "memoria_mb": [
            df[coluna].memory_usage(deep=True) / 1024**2
            for coluna in df.columns
        ],
    }
)

tipos_memoria = tipos_memoria.sort_values(
    "memoria_mb",
    ascending=False,
).reset_index(drop=True)

memoria_total_mb = (
    df.memory_usage(deep=True).sum()
    / 1024**2
)

print(f"Memória total: {memoria_total_mb:,.2f} MB")
display(tipos_memoria)

Memória total: 3,847.05 MB


,coluna,tipo_pandas,memoria_mb
0,nu_notific,str,92.276205
1,dt_notific,str,80.112946
2,dt_sin_pri,str,80.112946
3,dt_digita,str,79.958382
4,dt_interna,str,77.767781
...,...,...,...
83,out_anim,str,36.205858
84,dt_trt_cov,str,36.198720
85,out_trat,str,36.197567
86,tipo_trat,str,36.166728


In [37]:
# Datas que definem ou contextualizam o episódio de SRAG
COLUNAS_DATA_EPISODIO = [
    "dt_notific",
    "dt_sin_pri",
    "dt_interna",
]

# Datas históricas ou anteriores à admissão,
# cuja validade depende do contexto e da regra de aplicabilidade
COLUNAS_DATA_VACINACAO = [
    "dt_ut_dose",
    "dt_vac_mae",
    "dt_doseuni",
    "dt_1_dose",
    "dt_2_dose",
]

# Datas de tratamento com temporalidade a confirmar
COLUNAS_DATA_TRATAMENTO = [
    "dt_antivir",
    "dt_trt_cov",
]

# Datas posteriores ao episódio ou de caráter administrativo
COLUNAS_DATA_AUDITORIA = [
    "dt_evoluca",
    "dt_encerra",
    "dt_digita",
]

# Lista consolidada, sem duplicidades
COLUNAS_DATA = list(
    dict.fromkeys(
        COLUNAS_DATA_EPISODIO
        + COLUNAS_DATA_VACINACAO
        + COLUNAS_DATA_TRATAMENTO
        + COLUNAS_DATA_AUDITORIA
    )
)

print(f"Datas do episódio: {len(COLUNAS_DATA_EPISODIO)}")
print(f"Datas de vacinação: {len(COLUNAS_DATA_VACINACAO)}")
print(f"Datas de tratamento: {len(COLUNAS_DATA_TRATAMENTO)}")
print(f"Datas de auditoria: {len(COLUNAS_DATA_AUDITORIA)}")
print(f"Total de datas: {len(COLUNAS_DATA)}")

COLUNAS_DATA_EXISTENTES = [
    coluna
    for coluna in COLUNAS_DATA
    if coluna in df.columns
]

datas_convertidas = pd.DataFrame(
    {
        coluna: pd.to_datetime(
            df[coluna],
            errors="coerce",
        )
        for coluna in COLUNAS_DATA_EXISTENTES
    },
    index=df.index,
)

resumo_datas = pd.DataFrame(
    {
        "coluna": COLUNAS_DATA_EXISTENTES,
        "data_minima": [
            datas_convertidas[coluna].min()
            for coluna in COLUNAS_DATA_EXISTENTES
        ],
        "data_maxima": [
            datas_convertidas[coluna].max()
            for coluna in COLUNAS_DATA_EXISTENTES
        ],
        "validas": [
            datas_convertidas[coluna].notna().sum()
            for coluna in COLUNAS_DATA_EXISTENTES
        ],
        "invalidas_ou_ausentes": [
            datas_convertidas[coluna].isna().sum()
            for coluna in COLUNAS_DATA_EXISTENTES
        ],
    }
)

display(resumo_datas)

Datas do episódio: 3
Datas de vacinação: 5
Datas de tratamento: 2
Datas de auditoria: 3
Total de datas: 13


,coluna,data_minima,data_maxima,validas,invalidas_ou_ausentes
0,dt_notific,2018-12-30,2026-09-28,4666910,0
1,dt_sin_pri,2018-12-30,2026-09-26,4666910,0
2,dt_interna,1695-01-17,2224-08-10,4362665,304245
3,dt_ut_dose,2009-03-02,2026-09-23,390041,4276869
4,dt_vac_mae,1955-11-16,2026-09-21,5645,4661265
5,dt_doseuni,2013-04-25,2026-09-18,17601,4649309
6,dt_1_dose,2012-08-06,2026-09-22,18626,4648284
7,dt_2_dose,2012-09-14,2026-09-22,10895,4656015
8,dt_antivir,1717-09-27,2202-11-06,347712,4319198
9,dt_trt_cov,2020-03-15,2026-09-27,3833,4663077


In [38]:
if "dt_notific" not in datas_convertidas.columns:
    raise KeyError(
        "A coluna dt_notific não está disponível "
        "para avaliar o período temporal."
    )

ano_notificacao = (
    datas_convertidas["dt_notific"]
    .dt.year
    .rename("ano_notificacao")
)

volume_por_ano = (
    ano_notificacao
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("ano_notificacao")
    .reset_index(name="quantidade_registros")
)

volume_por_ano["percentual"] = (
    volume_por_ano["quantidade_registros"]
    / len(df)
    * 100
)

display(volume_por_ano)

,ano_notificacao,quantidade_registros,percentual
0,2018,3,0.000064
1,2019,48634,1.042103
2,2020,1148787,24.615581
3,2021,1773875,38.009625
4,2022,582656,12.484835
5,2023,284383,6.093604
6,2024,268519,5.753679
7,2025,334610,7.169840
8,2026,225443,4.830670


In [39]:
if "dt_interna" not in datas_convertidas.columns:
    raise KeyError(
        "A coluna dt_interna não está disponível "
        "para avaliar o período de internação."
    )

ano_internacao = (
    datas_convertidas["dt_interna"]
    .dt.year
    .rename("ano_internacao")
)

volume_por_ano_int = (
    ano_internacao
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("ano_internacao")
    .reset_index(name="quantidade_registros_int")
)

volume_por_ano_int["percentual"] = (
    volume_por_ano_int["quantidade_registros_int"]
    / len(df)
    * 100
)

display(volume_por_ano_int)

,ano_internacao,quantidade_registros_int,percentual
0,1695.0,8,0.000171
1,1717.0,2,0.000043
2,1752.0,2,0.000043
3,1755.0,1,0.000021
4,1772.0,3,0.000064
5,1852.0,2,0.000043
6,1853.0,1,0.000021
7,1854.0,1,0.000021
8,1864.0,1,0.000021
9,1917.0,4,0.000086


In [40]:
memoria_mb = (
    df.memory_usage(deep=True).sum()
    / 1024**2
)

print(f"Memória total utilizada: {memoria_mb:,.2f} MB")

Memória total utilizada: 3,847.05 MB


In [41]:
memoria_colunas = (
    df.memory_usage(deep=True)
    .div(1024**2)
    .rename("memoria_mb")
    .sort_values(ascending=False)
    .reset_index()
    .rename(columns={"index": "coluna"})
)

display(memoria_colunas)

,coluna,memoria_mb
0,nu_notific,92.276079
1,dt_sin_pri,80.112820
2,dt_notific,80.112820
3,dt_digita,79.958256
4,dt_interna,77.767655
...,...,...
84,dt_trt_cov,36.198594
85,out_trat,36.197441
86,tipo_trat,36.166602
87,tabag,36.162040


In [42]:
ausencias = pd.DataFrame(
    {
        "coluna": df.columns,
        "quantidade_ausente": [
            df[coluna].isna().sum()
            for coluna in df.columns
        ],
        "quantidade_preenchida": [
            df[coluna].notna().sum()
            for coluna in df.columns
        ],
    }
)

ausencias["percentual_ausente"] = (
    ausencias["quantidade_ausente"]
    / len(df)
    * 100
)

ausencias["percentual_preenchido"] = (
    ausencias["quantidade_preenchida"]
    / len(df)
    * 100
)

ausencias = ausencias.sort_values(
    "percentual_ausente",
    ascending=False,
).reset_index(drop=True)

display(ausencias)

,coluna,quantidade_ausente,quantidade_preenchida,percentual_ausente,percentual_preenchido
0,tabag,4666910,0,100.000000,0.000000
1,out_trat,4664116,2794,99.940132,0.059868
2,dt_trt_cov,4663077,3833,99.917869,0.082131
3,tipo_trat,4662126,4784,99.897491,0.102509
4,out_anim,4661634,5276,99.886949,0.113051
...,...,...,...,...,...
83,dt_notific,0,4666910,0.000000,100.000000
84,sem_pri,0,4666910,0.000000,100.000000
85,dt_sin_pri,0,4666910,0.000000,100.000000
86,cs_sexo,0,4666910,0.000000,100.000000


In [43]:
df_temporal = df.copy()

df_temporal["ano_notificacao"] = ano_notificacao

preenchimento_por_ano = (
    df_temporal
    .groupby("ano_notificacao", dropna=False)
    [COLUNAS_EDA]
    .apply(lambda grupo: grupo.notna().mean() * 100)
    .reset_index()
)

display(preenchimento_por_ano)

,ano_notificacao,nu_notific,dt_notific,dt_sin_pri,sem_pri,hospital,dt_interna,cs_sexo,nu_idade_n,tp_idade,...,antiviral,tp_antivir,dt_antivir,trat_cov,tipo_trat,dt_trt_cov,evolucao,dt_evoluca,dt_encerra,dt_digita
0,2018,100.0,100.0,100.0,100.0,100.000000,100.000000,100.0,100.0,100.0,...,100.000000,66.666667,66.666667,0.000000,0.000000,0.000000,100.000000,100.000000,100.000000,66.666667
1,2019,100.0,100.0,100.0,100.0,99.440720,97.318748,100.0,100.0,100.0,...,98.603857,60.762018,60.759962,0.193280,0.000000,0.000000,94.189250,84.081096,96.418144,65.419254
2,2020,100.0,100.0,100.0,100.0,97.624625,92.570772,100.0,100.0,100.0,...,86.346381,14.819022,14.513047,1.013939,0.016278,0.000783,93.641206,87.272401,94.055643,100.000000
3,2021,100.0,100.0,100.0,100.0,97.372476,93.659249,100.0,100.0,100.0,...,81.779043,2.246043,1.929223,1.078881,0.007329,0.002988,91.929025,85.798999,92.465534,100.000000
4,2022,100.0,100.0,100.0,100.0,96.706461,92.862856,100.0,100.0,100.0,...,79.995915,3.612766,3.271742,16.454821,0.171456,0.140735,91.922335,85.642815,91.871018,100.000000
5,2023,100.0,100.0,100.0,100.0,97.215023,94.232778,100.0,100.0,100.0,...,81.149365,5.872714,5.345256,74.884223,0.420208,0.318584,94.987745,88.153300,94.461694,100.000000
6,2024,100.0,100.0,100.0,100.0,97.840749,94.319955,100.0,100.0,100.0,...,81.128710,8.117861,7.524235,75.597258,0.359379,0.320275,94.417900,87.878325,93.960949,100.000000
7,2025,100.0,100.0,100.0,100.0,97.960910,94.575476,100.0,100.0,100.0,...,86.048833,11.776098,10.798542,79.988643,0.273453,0.249544,95.451720,86.436747,95.267326,100.000000
8,2026,100.0,100.0,100.0,100.0,97.635766,93.910212,100.0,100.0,100.0,...,87.339150,12.552175,11.804758,81.069716,0.174323,0.155250,86.790453,77.364123,86.690649,100.000000


In [44]:
cardinalidade = pd.DataFrame(
    {
        "coluna": df.columns,
        "valores_distintos_sem_na": [
            df[coluna].nunique(dropna=True)
            for coluna in df.columns
        ],
        "valores_distintos_com_na": [
            df[coluna].nunique(dropna=False)
            for coluna in df.columns
        ],
    }
)

cardinalidade = cardinalidade.sort_values(
    "valores_distintos_com_na",
    ascending=False,
).reset_index(drop=True)

display(cardinalidade)

,coluna,valores_distintos_sem_na,valores_distintos_com_na
0,nu_notific,4666881,4666881
1,outro_des,251506,251507
2,morb_desc,232631,232632
3,co_mun_res,5604,5605
4,co_mun_not,3661,3662
...,...,...,...
83,cs_sexo,3,3
84,tp_idade,3,3
85,estrang,2,3
86,pov_ct,2,3


In [45]:
LIMITE_CARDINALIDADE = 30

colunas_categoricas_baixas = cardinalidade.loc[
    cardinalidade["valores_distintos_com_na"]
    <= LIMITE_CARDINALIDADE,
    "coluna",
].tolist()

frequencias_categoricas = []

for coluna in colunas_categoricas_baixas:
    frequencia = (
        df[coluna]
        .value_counts(dropna=False)
        .rename_axis("valor")
        .reset_index(name="quantidade")
    )

    frequencia.insert(0, "coluna", coluna)

    frequencia["percentual"] = (
        frequencia["quantidade"]
        / len(df)
        * 100
    )

    frequencias_categoricas.append(frequencia)

frequencias_categoricas_df = pd.concat(
    frequencias_categoricas,
    ignore_index=True,
)

display(frequencias_categoricas_df.head(100))

,coluna,valor,quantidade,percentual
0,sg_uf_not,SP,1306611,27.997347
1,sg_uf_not,MG,521998,11.185088
2,sg_uf_not,RJ,385820,8.267140
3,sg_uf_not,PR,380639,8.156125
4,sg_uf_not,RS,256895,5.504606
...,...,...,...,...
95,cs_escol_n,3,382584,8.197801
96,cs_escol_n,5,337951,7.241430
97,cs_escol_n,2,250435,5.366184
98,cs_escol_n,0,213463,4.573969


In [46]:
if "nu_notific" in df.columns:
    identificadores_preenchidos = df["nu_notific"].notna().sum()

    identificadores_duplicados = (
        df.loc[
            df["nu_notific"].notna(),
            "nu_notific",
        ]
        .duplicated(keep=False)
        .sum()
    )

    resumo_identificador = pd.DataFrame(
        {
            "metrica": [
                "linhas_totais",
                "identificadores_preenchidos",
                "linhas_com_identificador_duplicado",
            ],
            "valor": [
                len(df),
                identificadores_preenchidos,
                identificadores_duplicados,
            ],
        }
    )

    display(resumo_identificador)
else:
    print(
        "nu_notific não foi carregada. "
        "A auditoria de identificador não será executada."
    )

,metrica,valor
0,linhas_totais,4666910
1,identificadores_preenchidos,4666910
2,linhas_com_identificador_duplicado,58


In [47]:
COLUNAS_POTENCIALMENTE_POS_ADMISSAO = [
    "evolucao",
    "dt_evoluca",
    "dt_encerra",
    "dt_digita",
]

auditoria_vazamento = pd.DataFrame(
    {
        "coluna": COLUNAS_POTENCIALMENTE_POS_ADMISSAO,
        "existe_no_dataframe": [
            coluna in df.columns
            for coluna in COLUNAS_POTENCIALMENTE_POS_ADMISSAO
        ],
        "uso_nesta_eda": "auditoria",
        "candidata_feature": False,
        "justificativa": (
            "Desfecho ou informação potencialmente "
            "posterior à admissão hospitalar."
        ),
    }
)

display(auditoria_vazamento)

,coluna,existe_no_dataframe,uso_nesta_eda,candidata_feature,justificativa
0,evolucao,True,auditoria,False,Desfecho ou informação potencialmente posterio...
1,dt_evoluca,True,auditoria,False,Desfecho ou informação potencialmente posterio...
2,dt_encerra,True,auditoria,False,Desfecho ou informação potencialmente posterio...
3,dt_digita,True,auditoria,False,Desfecho ou informação potencialmente posterio...


In [48]:
resumo_primeira_passagem = pd.DataFrame(
    {
        "metrica": [
            "versao_dados",
            "arquivo",
            "linhas_parquet",
            "colunas_parquet",
            "linhas_carregadas",
            "colunas_carregadas",
            "memoria_mb",
            "data_minima_notificacao",
            "data_maxima_notificacao",
            "colunas_ausentes",
        ],
        "valor": [
            C.DATA_VERSION,
            caminho_parquet.name,
            arquivo_parquet.metadata.num_rows,
            arquivo_parquet.metadata.num_columns,
            df.shape[0],
            df.shape[1],
            round(memoria_total_mb, 2),
            datas_convertidas["dt_notific"].min(),
            datas_convertidas["dt_notific"].max(),
            len(colunas_ausentes),
        ],
    }
)

display(resumo_primeira_passagem)

,metrica,valor
0,versao_dados,dados-v2
1,arquivo,srag.parquet
2,linhas_parquet,4666910
3,colunas_parquet,198
4,linhas_carregadas,4666910
5,colunas_carregadas,88
6,memoria_mb,3847.05
7,data_minima_notificacao,2018-12-30 00:00:00
8,data_maxima_notificacao,2026-09-28 00:00:00
9,colunas_ausentes,0


In [49]:
C.TABLES_DIR.mkdir(
parents=True,
exist_ok=True,
)

print(f"Tabelas serão salvas em: {C.TABLES_DIR}")

Tabelas serão salvas em: C:\Users\olivem85\OneDrive - Pfizer\Documents\GitHub\Projeto_Integrador\outputs\tables


In [51]:
tabelas_exportacao = {
    "eda_01_resumo_primeira_passagem.csv": (
        resumo_primeira_passagem
    ),
    "eda_01_status_colunas.csv": (
        status_colunas
    ),
    "eda_01_tipos_memoria.csv": (
        tipos_memoria
    ),
    "eda_01_resumo_datas.csv": (
        resumo_datas
    ),
    "eda_01_volume_por_ano.csv": (
        volume_por_ano
    ),
    "eda_01_ausencias.csv": (
        ausencias
    ),
    "eda_01_preenchimento_por_ano.csv": (
        preenchimento_por_ano
    ),
    "eda_01_cardinalidade.csv": (
        cardinalidade
    ),
    "eda_01_frequencias_categoricas.csv": (
        frequencias_categoricas_df
    ),
    "eda_01_auditoria_vazamento.csv": (
        auditoria_vazamento
    )
}   

for nome_arquivo, tabela in tabelas_exportacao.items():
    caminho_saida = C.TABLES_DIR / nome_arquivo

    tabela.to_csv(
        caminho_saida,
        index=False,
        encoding="utf-8-sig",
    )

    print(f"Salvo: {caminho_saida}")

Salvo: C:\Users\olivem85\OneDrive - Pfizer\Documents\GitHub\Projeto_Integrador\outputs\tables\eda_01_resumo_primeira_passagem.csv
Salvo: C:\Users\olivem85\OneDrive - Pfizer\Documents\GitHub\Projeto_Integrador\outputs\tables\eda_01_status_colunas.csv
Salvo: C:\Users\olivem85\OneDrive - Pfizer\Documents\GitHub\Projeto_Integrador\outputs\tables\eda_01_tipos_memoria.csv
Salvo: C:\Users\olivem85\OneDrive - Pfizer\Documents\GitHub\Projeto_Integrador\outputs\tables\eda_01_resumo_datas.csv
Salvo: C:\Users\olivem85\OneDrive - Pfizer\Documents\GitHub\Projeto_Integrador\outputs\tables\eda_01_volume_por_ano.csv
Salvo: C:\Users\olivem85\OneDrive - Pfizer\Documents\GitHub\Projeto_Integrador\outputs\tables\eda_01_ausencias.csv
Salvo: C:\Users\olivem85\OneDrive - Pfizer\Documents\GitHub\Projeto_Integrador\outputs\tables\eda_01_preenchimento_por_ano.csv
Salvo: C:\Users\olivem85\OneDrive - Pfizer\Documents\GitHub\Projeto_Integrador\outputs\tables\eda_01_cardinalidade.csv
Salvo: C:\Users\olivem85\OneDriv

In [52]:
verificacoes_finais = {
    "dataframe_nao_vazio": not df.empty,
    "colunas_solicitadas_presentes": not colunas_ausentes,
    "dt_notific_disponivel": "dt_notific" in df.columns,
    "evolucao_apenas_auditoria": (
        "evolucao" in COLUNAS_DESFECHO_AUDITORIA
        and "evolucao" not in COLUNAS_CANDIDATAS_PRINCIPAIS
    ),
    "diretorio_tabelas_existe": C.TABLES_DIR.exists(),
}

verificacoes_finais_df = pd.DataFrame(
    {
        "verificacao": verificacoes_finais.keys(),
        "resultado": verificacoes_finais.values(),
    }
)

display(verificacoes_finais_df)

if not all(verificacoes_finais.values()):
    falhas = [
        nome
        for nome, resultado in verificacoes_finais.items()
        if not resultado
    ]

    raise RuntimeError(
        "Verificações finais com falha: "
        f"{falhas}"
    )

print("Primeira passagem concluída com sucesso.")

,verificacao,resultado
0,dataframe_nao_vazio,True
1,colunas_solicitadas_presentes,True
2,dt_notific_disponivel,True
3,evolucao_apenas_auditoria,True
4,diretorio_tabelas_existe,True


Primeira passagem concluída com sucesso.


### Base utilizada

- Versão dos dados: dados-v2
- Nome do Parquet:srag.parquet
- Número de linhas: 4666910
- Número de colunas do Parquet: 198
- Número de colunas carregadas: 88
- Período observado em `dt_notific`: 2018 - 2026
- Memória utilizada: 3,847.05 mb

### Achados principais

- Colunas não encontradas: 0
- Colunas com maior ausência: tabag, out_trat, dt_trt_cov, tipo_trat, out_anim, dt_vac_mae, dt_2_dose, dt_doseuni, dt_1_dose, cs_etinia, tp_pov_ct, out_antiv, dose_adic, obes_imc, pac_cocbo, pac_dscbo, m_amamenta, dos_re_bi, mae_vac, dose_2ref, dt_antivir, tp_antivir, dt_ut_dose, dose_ref (todas cima de 90% de missing)
- Colunas com mudança aparente entre anos: cs_escol_n, diarreia, vomito, vacina_cov, dose_1_cov, dose_2_cov, estrang
- Datas que precisam de auditoria: dt_notific (2018), dt_sin_pri (2018), dt_interna (1695, 2224), dt_antivir (1717, 2202)
- Duplicidades que precisam de investigação: 58

#### Interpretação e encaminhamento

**Estrutura e construção da coorte**

- `nu_notific` será mantida para identificação técnica e investigação
  de duplicidades, não como preditora.
- `dt_notific`, `dt_sin_pri`, `hospital` e `dt_interna` serão utilizadas
  na definição temporal e na construção da coorte.
- `nu_idade_n` deve ser interpretada em conjunto com `tp_idade`.

**Variáveis candidatas**

- Permanecem candidatas as variáveis demográficas, sintomas,
  comorbidades, vacinação e contexto geográfico disponíveis na base.
- O preenchimento isoladamente não determinará a exclusão de uma
  variável.
- Variáveis com forte mudança entre anos serão avaliadas na EDA
  descritiva e nos experimentos de modelagem.

**Campos condicionais ou de uso secundário**

- Datas detalhadas de vacinação, ocupação, povos tradicionais,
  tratamentos e campos textuais exigem transformação ou análise
  específica antes do uso.
- Campos com preenchimento muito baixo poderão ser explorados, mas não
  entrarão automaticamente na primeira versão dos modelos.

**Auditoria e prevenção de vazamento**

- `evolucao` será utilizada como fonte para construção do alvo.
- `dt_evoluca`, `dt_encerra` e `dt_digita` serão mantidas apenas para
  auditoria.
- Tratamentos e respectivas datas serão excluídos da primeira versão
  dos preditores enquanto sua disponibilidade até a admissão não estiver
  demonstrada.

**Problemas encontrados**

- Foram identificadas datas de internação e antiviral fora de faixas
  plausíveis.
- Foram encontradas 58 linhas pertencentes a identificadores duplicados,
  que serão investigadas durante a construção da coorte.
- `tabag` está completamente vazio na versão atual.
- Algumas variáveis apresentam alta ausência ou mudança relevante de
  preenchimento entre anos.

**Próxima etapa**

- Construir uma EDA descritiva por ano e por grupos de variáveis.
- Preparar idade padronizada, recodificações categóricas e indicadores
  temporais.
- Definir a coorte analítica e o alvo binário.
- Iniciar baselines assim que o dataset analítico mínimo estiver pronto.
- Colunas essenciais: nu_notific, dt_notific, dt_sin_pri, nu_idade_n, tp_idade,cs_sexo, tosse, dispneia, garganta, desc_resp, saturacao, dor_abd, fadiga, perd_olft, asma, pneumopati, imunodepre, obesidade, out_morbi, morb_desc, outro_sin, outro_des, co_mun_not, co_mun_res, vacina_cov, antiviral